## Einsatz von KI

ChatGPT (OpenAI) wurde als unterstützendes Werkzeug bei der Erstellung
und Überarbeitung dieses Analyse-Notebooks verwendet. Die Unterstützung
umfasste insbesondere die Fehlersuche im Code, die Implementierung
statistischer Tests sowie die Klärung statistischer Methoden.

Alle Datenverarbeitungsschritte, statistischen Ergebnisse und Interpretationen wurden von uns überprüft und nachvollzogen.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

In [2]:
summary_path = "firestore_summary/results_summary.csv"

df = pd.read_csv(summary_path, sep=";")

df.head()

,participantId,counterbalanceGroup,deviceId,deviceModel,testerId,createdAt,consentAccepted,consentAcceptedAt,consentVersion,consentMethod,...,perceivedControl,perceivedEase,ownPhotoPreference,preferenceScaleMin,preferenceScaleMax,preferenceScaleLeft,preferenceScaleCenter,preferenceScaleRight,interactionEventCount,comment
0,P001,1,google_pixel_10_pro,Google Pixel 10 Pro,study_team,2026-10-06T12:30:54.799Z,True,2026-10-06T12:25:55.291Z,2026-10-06,in_app_checkbox,...,2,-2,1,-3,3,Swipe,Keine Präferenz / kein Unterschied,Raster,247,NaN
1,P003,3,samsung_galaxy_a36,Samsung Galaxy A36,study_team,2026-10-07T11:35:18.938Z,True,2026-10-07T11:29:09.703Z,2026-10-06,in_app_checkbox,...,0,1,3,-3,3,Swipe,Keine Präferenz / kein Unterschied,Raster,250,NaN
2,P004,4,google_pixel_10_pro,Google Pixel 10 Pro,study_team,2026-10-07T12:08:11.903Z,True,2026-10-07T12:03:11.333Z,2026-10-06,in_app_checkbox,...,2,-2,2,-3,3,Swipe,Keine Präferenz / kein Unterschied,Raster,259,NaN
3,P005,1,samsung_galaxy_a36,Samsung Galaxy A36,study_team,2026-10-07T15:18:15.470Z,True,2026-10-07T15:07:15.997Z,2026-10-06,in_app_checkbox,...,1,-3,0,-3,3,Swipe,Keine Präferenz / kein Unterschied,Raster,244,NaN
4,P006,2,google_pixel_10_pro,Google Pixel 10 Pro,study_team,2026-10-07T16:47:06.406Z,True,2026-10-07T16:33:18.680Z,2026-10-06,in_app_checkbox,...,3,1,1,-3,3,Swipe,Keine Präferenz / kein Unterschied,Raster,258,NaN


P002 fehlt weil die Anwendung einmal gestartet wurde ohne sie komplett abzuschließen

In [3]:
# Nachträglich bestätigtes Alter für P021 ergänzt (Hat er vergessen einzutragen, wir haben dann nochmal nachgefragt)
df.loc[df["participantId"] == "P021", "age"] = 24
df.loc[
    df["participantId"] == "P021",
    ["participantId", "age", "ageNotProvided"]
]

,participantId,age,ageNotProvided
19,P021,24.0,True


In [53]:
df["age"].isna().sum()

np.int64(0)

In [5]:
print("Anzahl Zeilen:", len(df))
print("Teilnehmer:", df["participantId"].nunique())

df["participantId"].tolist()

Anzahl Zeilen: 21
Teilnehmer: 21


['P001',
 'P003',
 'P004',
 'P005',
 'P006',
 'P007',
 'P008',
 'P009',
 'P010',
 'P011',
 'P012',
 'P013',
 'P014',
 'P015',
 'P016',
 'P017',
 'P018',
 'P019',
 'P020',
 'P021',
 'P022']

In [6]:
df.shape

(21, 80)

In [7]:
# Duplikate bei den Teilnehmer-IDs prüfen
duplicates = df[df["participantId"].duplicated(keep=False)]

print("doppelte IDs:")
display(duplicates[["participantId"]] if not duplicates.empty else "Keine Duplikate")

doppelte IDs:


'Keine Duplikate'

In [8]:
print(df["consentAccepted"].value_counts(dropna=False))

consentAccepted
True    21
Name: count, dtype: int64


In [9]:
expected_decisions = {
    "SA_decisions": 20,
    "RA_decisions": 20,
    "SB_decisions": 8,
    "RB_decisions": 8
}

for column, expected in expected_decisions.items():
    incorrect = df[df[column] != expected]

    print(f"{column}:")
    if incorrect.empty:
        print(f"  Alle Teilnehmer haben {expected} Entscheidungen.")
    else:
        display(incorrect[["participantId", column]])

SA_decisions:
  Alle Teilnehmer haben 20 Entscheidungen.
RA_decisions:
  Alle Teilnehmer haben 20 Entscheidungen.
SB_decisions:
  Alle Teilnehmer haben 8 Entscheidungen.
RB_decisions:
  Alle Teilnehmer haben 8 Entscheidungen.


In [10]:
missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)

missing

ageGroup    21
comment     18
dtype: int64

Agegroup ist hier noch eingetragen (Früherer Stand der App)

In [11]:
missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)

missing_relevant = missing.drop(
    labels=["ageGroup", "comment"],
    errors="ignore"
)

missing_relevant

Series([], dtype: int64)

# Stichprobe beschreiben

In [12]:
age = pd.to_numeric(df["age"], errors="coerce")

print("N:", age.count())
print("Mittelwert:", round(age.mean(), 2))
print("Median:", round(age.median(), 2))
print("Standardabweichung:", round(age.std(), 2))
print("Minimum:", age.min())
print("Maximum:", age.max())

N: 21
Mittelwert: 30.1
Median: 25.0
Standardabweichung: 12.86
Minimum: 18.0
Maximum: 68.0


In [13]:
def counts_with_percent(column):
    counts = df[column].value_counts(dropna=False)
    percentages = df[column].value_counts(dropna=False, normalize=True) * 100

    return pd.DataFrame({
        "n": counts,
        "percent": percentages.round(1)
    })

counts_with_percent("gender")

,n,percent
gender,,
Männlich,14,66.7
Weiblich,7,33.3


In [14]:
counts_with_percent("smartphoneOs")

,n,percent
smartphoneOs,,
iOS,13,61.9
Android,8,38.1


In [15]:
counts_with_percent("photoSortingFrequency")

,n,percent
photoSortingFrequency,,
Seltener,10,47.6
Mehrmals pro Monat,5,23.8
Mehrmals pro Woche,3,14.3
Nie,3,14.3


In [16]:
counts_with_percent("deviceModel")

,n,percent
deviceModel,,
Google Pixel 10 Pro,14,66.7
Samsung Galaxy A36,7,33.3


# Objektive Messwerte
## Deskriptive Statistik

In [17]:
objective_columns = [
    "SA_durationSec",
    "RA_durationSec",
    "SB_durationSec",
    "RB_durationSec",
    "SA_accuracyPercent",
    "RA_accuracyPercent",
    "SB_accuracyPercent",
    "RB_accuracyPercent"
]

# String in numeric konvertieren
df[objective_columns] = df[objective_columns].apply(
    pd.to_numeric,
    errors="coerce"
)

df[objective_columns].dtypes

SA_durationSec        float64
RA_durationSec        float64
SB_durationSec        float64
RB_durationSec        float64
SA_accuracyPercent      int64
RA_accuracyPercent      int64
SB_accuracyPercent    float64
RB_accuracyPercent    float64
dtype: object

In [18]:
objective_descriptives = pd.DataFrame({
    "Mean": df[objective_columns].mean(),
    "Median": df[objective_columns].median(),
    "SD": df[objective_columns].std(),
    "Min": df[objective_columns].min(),
    "Max": df[objective_columns].max()
}).round(2)

objective_descriptives

,Mean,Median,SD,Min,Max
SA_durationSec,33.11,33.04,11.67,18.44,63.92
RA_durationSec,23.58,19.18,13.07,5.48,51.75
SB_durationSec,84.35,77.24,32.47,46.57,174.95
RB_durationSec,47.28,39.40,26.03,21.28,132.39
SA_accuracyPercent,88.33,95.00,20.33,5.00,100.00
RA_accuracyPercent,76.67,90.00,26.61,5.00,100.00
SB_accuracyPercent,68.45,75.00,23.26,25.00,100.00
RB_accuracyPercent,65.48,75.00,26.19,12.50,100.00


In [19]:
df[objective_columns].isna().sum()

SA_durationSec        0
RA_durationSec        0
SB_durationSec        0
RB_durationSec        0
SA_accuracyPercent    0
RA_accuracyPercent    0
SB_accuracyPercent    0
RB_accuracyPercent    0
dtype: int64

In [20]:
df.loc[
    (df["SA_accuracyPercent"] == df["SA_accuracyPercent"].min()) |
    (df["RA_accuracyPercent"] == df["RA_accuracyPercent"].min()),
    [
        "participantId",
        "SA_accuracyPercent",
        "RA_accuracyPercent",
        "SB_accuracyPercent",
        "RB_accuracyPercent"
    ]
]

,participantId,SA_accuracyPercent,RA_accuracyPercent,SB_accuracyPercent,RB_accuracyPercent
4,P006,5,5,37.5,50.0


## Genauigkeit und Zeit

## Gepaarte Differenzen

Da wir ein Within-Subject-Design verwenden, hat jede VP sowohl die Swipe- als auch die Grid-Bedingung für beide Aufgabentypen durchgeführt. Die Messwerte sind daher voneinander abhängig bzw. gepaart.

Für jede VP berechnen wir deshalb die Differenz zwischen Swipe und Grid getrennt für Bearbeitungszeit und Genauigkeit.

Die Differenzen werden jeweils als `Swipe - Grid` berechnet.

Diese Differenzwerte benötigen wir anschließend für den Shapiro-Wilk-Test.
Beim paired t-Test muss nicht jede einzelne Bedingung normalverteilt sein, sondern die Verteilung der Differenzen zwischen den gepaarten Messwerten.

In [21]:
# Gepaarte Differenzen: Swipe - Grid
df["diff_individual_duration"] = (
    df["SA_durationSec"] - df["RA_durationSec"]
)

df["diff_series_duration"] = (
    df["SB_durationSec"] - df["RB_durationSec"]
)

df["diff_individual_accuracy"] = (
    df["SA_accuracyPercent"] - df["RA_accuracyPercent"]
)

df["diff_series_accuracy"] = (
    df["SB_accuracyPercent"] - df["RB_accuracyPercent"]
)

### Interpretation der Differenzen

Bei der Bearbeitungszeit bedeutet ein negativer Wert, dass Swipe schneller
war als Grid. Ein positiver Wert bedeutet, dass Grid schneller
war.

Bei der Genauigkeit bedeutet ein positiver Wert, dass Swipe eine höhere
Genauigkeit erreicht hat. Ein negativer Wert bedeutet, dass Grid genauer war.

## Shapiro-Wilk-Test auf Normalverteilung

Bevor ein gepaarter t-Test durchgeführt wird, wird geprüft, ob die
Differenzwerte zwischen Swipe und Grid annähernd normalverteilt sind.

Dafür verwenden wir den Shapiro-Wilk-Test.

- p > 0.05: kein signifikanter Hinweis gegen eine Normalverteilung
- p <= 0.05: Normalverteilung wird verworfen

Bei normalverteilten Differenzen machen anschließend einen
gepaarten t-Test. Andernfalls machen wir den Wilcoxon-Signed-Rank-Test.

In [22]:
difference_columns = {
    "Individual Photos - Duration": "diff_individual_duration",
    "Photo Series - Duration": "diff_series_duration",
    "Individual Photos - Accuracy": "diff_individual_accuracy",
    "Photo Series - Accuracy": "diff_series_accuracy"
}

for label, column in difference_columns.items():
    result = stats.shapiro(df[column])

    print(label)
    print(f"W = {result.statistic:.3f}")
    print(f"p = {result.pvalue:.3f}")

    if result.pvalue > 0.05:
        print("→ Kein signifikanter Hinweis gegen Normalverteilung")
    else:
        print("→ Normalverteilung wird verworfen")

    print()

Individual Photos - Duration
W = 0.960
p = 0.518
→ Kein signifikanter Hinweis gegen Normalverteilung

Photo Series - Duration
W = 0.976
p = 0.867
→ Kein signifikanter Hinweis gegen Normalverteilung

Individual Photos - Accuracy
W = 0.939
p = 0.207
→ Kein signifikanter Hinweis gegen Normalverteilung

Photo Series - Accuracy
W = 0.972
p = 0.774
→ Kein signifikanter Hinweis gegen Normalverteilung



## Gepaarte t-Tests

Weil die Shapiro-Wilk-Tests für alle vier Differenzvariablen keinen signifikanten Hinweis auf eine Abweichung von der Normalverteilung zeigen, werden für die objektiven Maße paired t-Tests verwendet.

Verglichen werden jeweils Swipe und Grid innerhalb desselben Aufgabentyps.

Zusätzlich berechnen wir Cohen's dz als Effektgröße für die gepaarten Vergleiche.

In [23]:
comparisons = {
    "Individual Photos - Duration": (
        "SA_durationSec",
        "RA_durationSec"
    ),
    "Photo Series - Duration": (
        "SB_durationSec",
        "RB_durationSec"
    ),
    "Individual Photos - Accuracy": (
        "SA_accuracyPercent",
        "RA_accuracyPercent"
    ),
    "Photo Series - Accuracy": (
        "SB_accuracyPercent",
        "RB_accuracyPercent"
    )
}

results = []

for label, (swipe_col, grid_col) in comparisons.items():

    swipe = df[swipe_col]
    grid = df[grid_col]

    # Gepaarter t-Test
    test = stats.ttest_rel(swipe, grid)

    # Differenz: Swipe - Grid
    diff = swipe - grid

    # Cohens dz für gepaarte Daten
    cohens_dz = diff.mean() / diff.std(ddof=1)

    results.append({
        "Comparison": label,
        "Swipe Mean": swipe.mean(),
        "Grid Mean": grid.mean(),
        "Mean Difference": diff.mean(),
        "t": test.statistic,
        "df": len(diff) - 1,
        "p": test.pvalue,
        "Cohen's dz": cohens_dz
    })

ttest_results = pd.DataFrame(results)

ttest_results.round(3)

,Comparison,Swipe Mean,Grid Mean,Mean Difference,t,df,p,Cohen's dz
0,Individual Photos - Duration,33.106,23.582,9.524,2.983,20,0.007,0.651
1,Photo Series - Duration,84.347,47.279,37.068,4.727,20,0.000,1.031
2,Individual Photos - Accuracy,88.333,76.667,11.667,2.331,20,0.030,0.509
3,Photo Series - Accuracy,68.452,65.476,2.976,0.560,20,0.581,0.122


## Korrektur für multiples Testen

Weil vier statistische Tests für die objektiven Messgrößen durchgeführt wurden, steigt die Wahrscheinlichkeit, zufällig ein signifikantes Ergebnis zu erhalten.

Daher machen wir eine Bonferroni-Korrektur für diese vier Tests. Das ursprüngliche Signifikanzniveau von α = 0.05 wird durch die Anzahl der Tests geteilt:

α_adjusted = 0.05 / 4 = 0.0125

Zusätzlich werden Bonferroni-korrigierte p-Werte berechnet.

In [24]:
# Bonferroni-Korrektur für die vier objektiven Tests

number_of_tests = len(ttest_results)

alpha = 0.05
alpha_bonferroni = alpha / number_of_tests

ttest_results["p_bonferroni"] = (
    ttest_results["p"] * number_of_tests
).clip(upper=1)

ttest_results["significant_bonferroni"] = (
    ttest_results["p_bonferroni"] < alpha
)

print("Anzahl Tests:", number_of_tests)
print("Bonferroni-korrigiertes Alpha:", alpha_bonferroni)

ttest_results[
    [
        "Comparison",
        "t",
        "df",
        "p",
        "p_bonferroni",
        "Cohen's dz",
        "significant_bonferroni"
    ]
].round(3)

Anzahl Tests: 4
Bonferroni-korrigiertes Alpha: 0.0125


,Comparison,t,df,p,p_bonferroni,Cohen's dz,significant_bonferroni
0,Individual Photos - Duration,2.983,20,0.007,0.029,0.651,True
1,Photo Series - Duration,4.727,20,0.000,0.001,1.031,True
2,Individual Photos - Accuracy,2.331,20,0.030,0.121,0.509,False
3,Photo Series - Accuracy,0.560,20,0.581,1.000,0.122,False


## Sensitivitätsanalyse der objektiven Vergleiche

Da vorallem die Accuracy-Werte auf einer begrenzten Anzahl diskreter Entscheidungen basieren, werden die vier objektiven Swipe-Grid-Vergleiche zusätzlich mit Wilcoxon-Signed-Rank-Tests geprüft.

Diese Analyse ersetzt nicht die paired t-Tests, sondern dient als Robustheitsprüfung. Für die vier zusätzlichen Tests wird auch eine Bonferroni-Korrektur angewendet.


In [25]:
# Sensitivitätsanalyse für die vier objektiven Vergleiche

sensitivity_results = []

for label, (swipe_col, grid_col) in comparisons.items():
    test = stats.wilcoxon(
        df[swipe_col],
        df[grid_col],
        alternative="two-sided",
        method="auto"
    )

    sensitivity_results.append({
        "Comparison": label,
        "W": test.statistic,
        "p": test.pvalue
    })

sensitivity_results = pd.DataFrame(sensitivity_results)

# Bonferroni-Korrektur über dieselben vier Vergleichstests
sensitivity_results["p_bonferroni"] = (
    sensitivity_results["p"] * len(sensitivity_results)
).clip(upper=1)

sensitivity_results["significant_bonferroni"] = (
    sensitivity_results["p_bonferroni"] < 0.05
)

sensitivity_results.round(3)


,Comparison,W,p,p_bonferroni,significant_bonferroni
0,Individual Photos - Duration,40.0,0.007,0.028,True
1,Photo Series - Duration,15.0,0.000,0.001,True
2,Individual Photos - Accuracy,38.0,0.038,0.152,False
3,Photo Series - Accuracy,71.5,0.535,1.000,False


# Subjektive Bewertungen

Nach jeder Bedingung bewerteten die Teilnehmenden die Interaktion auf einer siebenstufigen Likert-Skala.

Erfasst wurden:
- Ease of use
- Perceived control
- Confidence
- Workload
- Satisfaction
- Playfulness

Bei den Foto-Serien wurde zusätzlich die wahrgenommene Vergleichbarkeit der Bilder abgefragt.

Da es sich um einzelne ordinale Daten handelt und dieselben Personen jeweils Swipe und Grid bewertet haben, werden die Bedingungen mit dem Wilcoxon-Signed-Rank-Test verglichen.

In [26]:
likert_items = [
    "ease",
    "control",
    "confidence",
    "workload",
    "satisfaction",
    "playfulness"
]

likert_descriptives = []

for item in likert_items:
    for condition in ["SA", "RA", "SB", "RB"]:
        column = f"{condition}_{item}"

        likert_descriptives.append({
            "Condition": condition,
            "Item": item,
            "Median": df[column].median(),
            "Q1": df[column].quantile(0.25),
            "Q3": df[column].quantile(0.75)
        })

likert_descriptives = pd.DataFrame(likert_descriptives)

likert_descriptives

,Condition,Item,Median,Q1,Q3
0,SA,ease,7.0,7.0,7.0
1,RA,ease,7.0,6.0,7.0
2,SB,ease,7.0,6.0,7.0
3,RB,ease,7.0,7.0,7.0
4,SA,control,7.0,6.0,7.0
5,RA,control,7.0,7.0,7.0
6,SB,control,7.0,5.0,7.0
7,RB,control,7.0,7.0,7.0
8,SA,confidence,7.0,6.0,7.0
9,RA,confidence,7.0,6.0,7.0


## Wilcoxon-Signed-Rank-Tests

Für jedes subjektive Item werden Swipe und Grid getrennt für die beiden Aufgabentypen verglichen.

- Individual Photos: SA vs. RA
- Photo Series: SB vs. RB

In [27]:
likert_results = []

for item in likert_items:

    # Individual Photos
    test_individual = stats.wilcoxon(
        df[f"SA_{item}"],
        df[f"RA_{item}"]
    )

    likert_results.append({
        "Task": "Individual Photos",
        "Item": item,
        "Swipe Median": df[f"SA_{item}"].median(),
        "Grid Median": df[f"RA_{item}"].median(),
        "W": test_individual.statistic,
        "p": test_individual.pvalue
    })

    # Photo Series
    test_series = stats.wilcoxon(
        df[f"SB_{item}"],
        df[f"RB_{item}"]
    )

    likert_results.append({
        "Task": "Photo Series",
        "Item": item,
        "Swipe Median": df[f"SB_{item}"].median(),
        "Grid Median": df[f"RB_{item}"].median(),
        "W": test_series.statistic,
        "p": test_series.pvalue
    })

likert_results = pd.DataFrame(likert_results)

# Zusätzliches Item nur für Photo Series: Vergleichbarkeit

comparison_test = stats.wilcoxon(
    df["SB_comparison"],
    df["RB_comparison"]
)

comparison_row = pd.DataFrame([{
    "Task": "Photo Series",
    "Item": "comparison",
    "Swipe Median": df["SB_comparison"].median(),
    "Grid Median": df["RB_comparison"].median(),
    "W": comparison_test.statistic,
    "p": comparison_test.pvalue
}])

likert_results = pd.concat(
    [likert_results, comparison_row],
    ignore_index=True
)

likert_results.round(3)

,Task,Item,Swipe Median,Grid Median,W,p
0,Individual Photos,ease,7.0,7.0,15.5,0.392
1,Photo Series,ease,7.0,7.0,1.5,0.031
2,Individual Photos,control,7.0,7.0,20.5,0.809
3,Photo Series,control,7.0,7.0,0.0,0.007
4,Individual Photos,confidence,7.0,7.0,16.5,0.831
5,Photo Series,confidence,5.0,7.0,8.0,0.002
6,Individual Photos,workload,1.0,1.0,11.5,0.667
7,Photo Series,workload,1.0,1.0,14.0,0.166
8,Individual Photos,satisfaction,7.0,7.0,29.0,0.417
9,Photo Series,satisfaction,6.0,7.0,23.5,0.117


## Korrektur für multiples Testen

Da mehrere subjektive Items innerhalb jedes Aufgabentyps getestet werden, wird eine Bonferroni-Korrektur getrennt für die Individual-Photo- und Photo-Series-Analysen durchgeführt.

Für Individual Photos werden sechs Tests berücksichtigt, für Photo Series sieben Tests.

In [28]:
likert_results["p_bonferroni"] = np.nan

for task in likert_results["Task"].unique():

    mask = likert_results["Task"] == task
    number_of_tests = mask.sum()

    likert_results.loc[mask, "p_bonferroni"] = (
        likert_results.loc[mask, "p"] * number_of_tests
    ).clip(upper=1)

likert_results["significant_bonferroni"] = (
    likert_results["p_bonferroni"] < 0.05
)

likert_results.round(3)

,Task,Item,Swipe Median,Grid Median,W,p,p_bonferroni,significant_bonferroni
0,Individual Photos,ease,7.0,7.0,15.5,0.392,1.000,False
1,Photo Series,ease,7.0,7.0,1.5,0.031,0.218,False
2,Individual Photos,control,7.0,7.0,20.5,0.809,1.000,False
3,Photo Series,control,7.0,7.0,0.0,0.007,0.046,True
4,Individual Photos,confidence,7.0,7.0,16.5,0.831,1.000,False
5,Photo Series,confidence,5.0,7.0,8.0,0.002,0.012,True
6,Individual Photos,workload,1.0,1.0,11.5,0.667,1.000,False
7,Photo Series,workload,1.0,1.0,14.0,0.166,1.000,False
8,Individual Photos,satisfaction,7.0,7.0,29.0,0.417,1.000,False
9,Photo Series,satisfaction,6.0,7.0,23.5,0.117,0.820,False


### Paarweise Verteilung der Kontrollbewertungen bei Foto-Serien

Weil der Median der wahrgenommenen Kontrolle sowohl für Swipe als auch für Grid bei 7 liegt, schauen wir zusätzlich auf die individuellen Paarunterschiede. Dadurch wird sichtbar, in welche Richtung sich die Bewertungen trotz identischer Mediane unterscheiden.

Zusätzlich wird eine rank biseriale Korrelation als Effektstärke für die paarweisen ordinalen Unterschiede berechnet.


In [29]:
# Paarweise Unterschiede: Grid (RB) - Swipe (SB)
control_diff = df["RB_control"] - df["SB_control"]

grid_higher = (control_diff > 0).sum()
equal_rating = (control_diff == 0).sum()
swipe_higher = (control_diff < 0).sum()

print("Grid höher bewertet:", grid_higher)
print("Gleich bewertet:", equal_rating)
print("Swipe höher bewertet:", swipe_higher)

# Rank-biseriale Korrelation auf Basis der Nicht-Null-Differenzen
nonzero_diff = control_diff[control_diff != 0]

if len(nonzero_diff) > 0:
    ranks = stats.rankdata(np.abs(nonzero_diff))
    r_plus = ranks[nonzero_diff > 0].sum()
    r_minus = ranks[nonzero_diff < 0].sum()
    rank_biserial = (r_plus - r_minus) / (r_plus + r_minus)
else:
    rank_biserial = np.nan

print("Rank-biseriale Korrelation:", round(rank_biserial, 3))

# Möglichen Ceiling Effect sichtbar machen
print("Swipe-Control = 7:", (df["SB_control"] == 7).sum())
print("Grid-Control = 7:", (df["RB_control"] == 7).sum())


Grid höher bewertet: 9
Gleich bewertet: 12
Swipe höher bewertet: 0
Rank-biseriale Korrelation: 1.0
Swipe-Control = 7: 11
Grid-Control = 7: 16


# Abschließender Präferenzfragebogen

Am Ende der Studie vergleichen die Teilnehmenden Swipe und Grid direkt auf einer bipolaren Skala von -3 bis +3.

Negative Werte sprechen für Swipe, positive Werte für Grid und 0 entspricht keiner Präferenz bzw. keinem wahrgenommenen Unterschied.

Da die Skala ordinal ist, werden die Antworten mit einem Wilcoxon-Signed-Rank-Test gegen den neutralen Mittelpunkt 0 getestet.

In [30]:
preference_items = [
    "overallPreference",
    "independentPreference",
    "seriesPreference",
    "perceivedSpeed",
    "perceivedControl",
    "perceivedEase",
    "ownPhotoPreference"
]

preference_results = []

for item in preference_items:
    values = df[item]

    test = stats.wilcoxon(values)

    preference_results.append({
        "Item": item,
        "Median": values.median(),
        "Q1": values.quantile(0.25),
        "Q3": values.quantile(0.75),
        "W": test.statistic,
        "p": test.pvalue
    })

preference_results = pd.DataFrame(preference_results)

preference_results.round(3)

,Item,Median,Q1,Q3,W,p
0,overallPreference,0.0,-1.0,2.0,60.0,0.427
1,independentPreference,2.0,-1.0,3.0,58.0,0.075
2,seriesPreference,2.0,-1.0,3.0,53.0,0.049
3,perceivedSpeed,0.0,-3.0,3.0,103.5,0.953
4,perceivedControl,2.0,0.0,3.0,2.5,0.001
5,perceivedEase,0.0,-2.0,2.0,94.5,0.691
6,ownPhotoPreference,2.0,1.0,3.0,14.5,0.002


In [31]:
# Bonferroni-Korrektur für die 7 finalen Präferenzfragen

number_of_tests = len(preference_results)

preference_results["p_bonferroni"] = (
    preference_results["p"] * number_of_tests
).clip(upper=1)

preference_results["significant_bonferroni"] = (
    preference_results["p_bonferroni"] < 0.05
)

preference_results.round(3)

,Item,Median,Q1,Q3,W,p,p_bonferroni,significant_bonferroni
0,overallPreference,0.0,-1.0,2.0,60.0,0.427,1.000,False
1,independentPreference,2.0,-1.0,3.0,58.0,0.075,0.523,False
2,seriesPreference,2.0,-1.0,3.0,53.0,0.049,0.340,False
3,perceivedSpeed,0.0,-3.0,3.0,103.5,0.953,1.000,False
4,perceivedControl,2.0,0.0,3.0,2.5,0.001,0.004,True
5,perceivedEase,0.0,-2.0,2.0,94.5,0.691,1.000,False
6,ownPhotoPreference,2.0,1.0,3.0,14.5,0.002,0.012,True


# Explorative Analyse: Easy vs. Hard bei Foto-Serien

Die Easy-Serien wurden in der Studie immer vor den Hard-Serien präsentiert. Damit sind Schwierigkeitsgrad und Reihenfolge miteinander vermischt. Die folgenden Analysen prüfen daher, ob sich die Swipe-Grid-Unterschiede zwischen den als Easy und Hard klassifizierten Serien verändern, werden aber nur vorsichtig interpretiert und nicht als eindeutiger kausaler Effekt der Schwierigkeit verstanden.


In [32]:
events_path = "firestore_summary/interaction_events.csv"

events = pd.read_csv(events_path, sep=";")

print("Anzahl Events:", len(events))
print("Teilnehmer:", events["participantId"].nunique())

events.head()

Anzahl Events: 5347
Teilnehmer: 21


,participantId,counterbalanceGroup,deviceId,deviceModel,sequence,timestamp,conditionId,interfaceType,taskType,eventType,...,gridPosition,action,previousAction,previousPhotoId,durationMs,durationSec,visibleDurationMs,visibleDurationSec,expectedAction,isBestInSeries
0,P001,1,google_pixel_10_pro,Google Pixel 10 Pro,1,2026-10-06T12:26:06.494Z,SA,swipe,independent,photo_shown,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,keep,NaN
1,P001,1,google_pixel_10_pro,Google Pixel 10 Pro,2,2026-10-06T12:26:12.681Z,SA,swipe,independent,photo_action,...,NaN,keep,NaN,NaN,6186.0,6.186,NaN,NaN,keep,NaN
2,P001,1,google_pixel_10_pro,Google Pixel 10 Pro,3,2026-10-06T12:26:12.687Z,SA,swipe,independent,photo_shown,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,keep,NaN
3,P001,1,google_pixel_10_pro,Google Pixel 10 Pro,4,2026-10-06T12:26:14.051Z,SA,swipe,independent,photo_action,...,NaN,keep,NaN,NaN,1364.0,1.364,NaN,NaN,keep,NaN
4,P001,1,google_pixel_10_pro,Google Pixel 10 Pro,5,2026-10-06T12:26:14.054Z,SA,swipe,independent,photo_shown,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,keep,NaN


In [33]:
series_events = events[
    events["conditionId"].isin(["SB", "RB"])
].copy()

print("Events in den Serienbedingungen:", len(series_events))

print("\nSchwierigkeiten:")
print(series_events["difficulty"].value_counts(dropna=False))

print("\nEvents nach Bedingung und Schwierigkeit:")
display(
    series_events
    .groupby(["conditionId", "difficulty"])
    .size()
    .reset_index(name="Events")
)

Events in den Serienbedingungen: 3556

Schwierigkeiten:
difficulty
easy    1831
hard    1725
Name: count, dtype: int64

Events nach Bedingung und Schwierigkeit:


,conditionId,difficulty,Events
0,RB,easy,843
1,RB,hard,845
2,SB,easy,988
3,SB,hard,880


In [34]:
display(
    series_events
    .groupby(["conditionId", "eventType"])
    .size()
    .reset_index(name="Anzahl")
)

,conditionId,eventType,Anzahl
0,RB,grid_confirmed,168
1,RB,grid_selection_changed,176
2,RB,photo_exposure_summary,672
3,RB,photo_shown,672
4,SB,photo_action,934
5,SB,photo_shown,934


In [35]:
columns_to_show = [
    "participantId",
    "conditionId",
    "interfaceType",
    "eventType",
    "seriesId",
    "difficulty",
    "action",
    "durationSec",
    "visibleDurationSec",
    "isBestInSeries"
]

display(series_events[columns_to_show].head(40))

,participantId,conditionId,interfaceType,eventType,seriesId,difficulty,action,durationSec,visibleDurationSec,isBestInSeries
91,P001,RB,grid,photo_shown,b2_e_s03,easy,NaN,NaN,NaN,True
92,P001,RB,grid,photo_shown,b2_e_s03,easy,NaN,NaN,NaN,False
93,P001,RB,grid,photo_shown,b2_e_s03,easy,NaN,NaN,NaN,False
94,P001,RB,grid,photo_shown,b2_e_s03,easy,NaN,NaN,NaN,False
95,P001,RB,grid,grid_selection_changed,b2_e_s03,easy,select,7.105,7.105,True
96,P001,RB,grid,photo_exposure_summary,b2_e_s03,easy,NaN,7.736,7.736,True
97,P001,RB,grid,photo_exposure_summary,b2_e_s03,easy,NaN,7.736,7.736,False
98,P001,RB,grid,photo_exposure_summary,b2_e_s03,easy,NaN,7.736,7.736,False
99,P001,RB,grid,photo_exposure_summary,b2_e_s03,easy,NaN,7.736,7.736,False
100,P001,RB,grid,grid_confirmed,b2_e_s03,easy,confirm_selection,NaN,NaN,True


## Auswertung auf Serienebene

Für die Analyse von Easy- und Hard-Serien werden die einzelnen Interaktionsereignisse zuerst zu einer Zeile pro bearbeiteter Foto-Serie zusammengefasst.

Die Bearbeitungszeit einer Serie wird als Zeit zwischen dem ersten geloggten Ereignis der Serie und dem letzten Ereignis der Serie berechnet.

Für Grid entspricht die finale Auswahl dem `grid_confirmed`-Ereignis. Für Swipe wird das zuletzt behaltene Foto einer Serie als finale Auswahl verwendet.

In [36]:
# Zeitstempel in echtes Datums/Zeitformat umwandeln
series_events["timestamp_dt"] = pd.to_datetime(
    series_events["timestamp"],
    utc=True
)

# Bearbeitungszeit pro Serie
series_time = (
    series_events
    .groupby(
        ["participantId", "conditionId", "seriesId", "difficulty"],
        as_index=False
    )
    .agg(
        start=("timestamp_dt", "min"),
        end=("timestamp_dt", "max")
    )
)

series_time["durationSec"] = (
    series_time["end"] - series_time["start"]
).dt.total_seconds()



In [37]:
# GRID
# grid_confirmed enthält die finale Auswahl der Serie

grid_accuracy = series_events[
    (series_events["conditionId"] == "RB") &
    (series_events["eventType"] == "grid_confirmed")
][
    [
        "participantId",
        "conditionId",
        "seriesId",
        "difficulty",
        "isBestInSeries"
    ]
].copy()

grid_accuracy["correct"] = (
    grid_accuracy["isBestInSeries"]
    .astype(str)
    .str.lower()
    .eq("true")
)

grid_accuracy = grid_accuracy.drop(columns="isBestInSeries")

In [38]:
# SWIPE
# Die finale Auswahl entspricht dem letzten behaltenen Foto der Serie

swipe_keeps = series_events[
    (series_events["conditionId"] == "SB") &
    (series_events["eventType"] == "photo_action") &
    (series_events["action"] == "keep")
].copy()

swipe_keeps = swipe_keeps.sort_values(
    ["participantId", "seriesId", "sequence"]
)

swipe_accuracy = (
    swipe_keeps
    .groupby(
        ["participantId", "conditionId", "seriesId", "difficulty"],
        as_index=False
    )
    .tail(1)
    .copy()
)

swipe_accuracy["correct"] = (
    swipe_accuracy["isBestInSeries"]
    .astype(str)
    .str.lower()
    .eq("true")
)

swipe_accuracy = swipe_accuracy[
    [
        "participantId",
        "conditionId",
        "seriesId",
        "difficulty",
        "correct"
    ]
]

In [39]:
series_accuracy = pd.concat(
    [swipe_accuracy, grid_accuracy],
    ignore_index=True
)

series_level = series_time.merge(
    series_accuracy,
    on=[
        "participantId",
        "conditionId",
        "seriesId",
        "difficulty"
    ],
    how="left"
)

display(series_level.head(10))

print("Anzahl Serien:", len(series_level))
print("Fehlende Accuracy-Werte:", series_level["correct"].isna().sum())

display(
    series_level
    .groupby(["conditionId", "difficulty"])
    .size()
    .reset_index(name="Serien")
)

,participantId,conditionId,seriesId,difficulty,start,end,durationSec,correct
0,P001,RB,b2_e_s01,easy,2026-10-06 12:28:14.018000+00:00,2026-10-06 12:28:15.871000+00:00,1.853,True
1,P001,RB,b2_e_s02,easy,2026-10-06 12:28:06.937000+00:00,2026-10-06 12:28:11.341000+00:00,4.404,False
2,P001,RB,b2_e_s03,easy,2026-10-06 12:27:59.169000+00:00,2026-10-06 12:28:06.905000+00:00,7.736,True
3,P001,RB,b2_e_s04,easy,2026-10-06 12:28:11.378000+00:00,2026-10-06 12:28:13.986000+00:00,2.608,True
4,P001,RB,b2_h_s01,hard,2026-10-06 12:28:24.486000+00:00,2026-10-06 12:28:28.135000+00:00,3.649,True
5,P001,RB,b2_h_s02,hard,2026-10-06 12:28:15.898000+00:00,2026-10-06 12:28:18.458000+00:00,2.560,True
6,P001,RB,b2_h_s03,hard,2026-10-06 12:28:21.100000+00:00,2026-10-06 12:28:24.459000+00:00,3.359,True
7,P001,RB,b2_h_s04,hard,2026-10-06 12:28:18.501000+00:00,2026-10-06 12:28:21.070000+00:00,2.569,True
8,P001,SB,b1_e_s01,easy,2026-10-06 12:28:54.528000+00:00,2026-10-06 12:28:58.921000+00:00,4.393,True
9,P001,SB,b1_e_s02,easy,2026-10-06 12:28:58.922000+00:00,2026-10-06 12:29:03.014000+00:00,4.092,True


Anzahl Serien: 336
Fehlende Accuracy-Werte: 0


,conditionId,difficulty,Serien
0,RB,easy,84
1,RB,hard,84
2,SB,easy,84
3,SB,hard,84


## Zusammenfassung nach Schwierigkeit

Für jede VP werden die vier Easy- und vier Hard-Serien innerhalb jeder Interaktionstechnik zusammengefasst.

Für jede Kombination aus Interaktionstechnik und Schwierigkeit berechnen wir:

- die mittlere Bearbeitungszeit pro Serie
- die Accuracy über die vier Serien

In [40]:
series_summary = (
    series_level
    .groupby(
        ["participantId", "conditionId", "difficulty"],
        as_index=False
    )
    .agg(
        mean_durationSec=("durationSec", "mean"),
        accuracyPercent=("correct", lambda x: x.mean() * 100),
        n_series=("seriesId", "nunique")
    )
)

display(series_summary.head(12))

print("Anzahl Zeilen:", len(series_summary))

display(
    series_summary["n_series"].value_counts()
)

,participantId,conditionId,difficulty,mean_durationSec,accuracyPercent,n_series
0,P001,RB,easy,4.15025,75.0,4
1,P001,RB,hard,3.03425,100.0,4
2,P001,SB,easy,5.50500,100.0,4
3,P001,SB,hard,6.13075,75.0,4
4,P003,RB,easy,4.80825,100.0,4
5,P003,RB,hard,3.26200,75.0,4
6,P003,SB,easy,17.45900,75.0,4
7,P003,SB,hard,5.27350,75.0,4
8,P004,RB,easy,4.21175,50.0,4
9,P004,RB,hard,3.95225,25.0,4


Anzahl Zeilen: 84


n_series
4    84
Name: count, dtype: int64

In [41]:
duration_easy_hard = series_summary.pivot(
    index="participantId",
    columns=["conditionId", "difficulty"],
    values="mean_durationSec"
)

accuracy_easy_hard = series_summary.pivot(
    index="participantId",
    columns=["conditionId", "difficulty"],
    values="accuracyPercent"
)

In [42]:
display(duration_easy_hard.head())
display(accuracy_easy_hard.head())

conditionId          RB                  SB          
difficulty         easy      hard      easy      hard
participantId                                        
P001            4.15025   3.03425   5.50500   6.13075
P003            4.80825   3.26200  17.45900   5.27350
P004            4.21175   3.95225  10.61375   3.48075
P005           21.22775  11.76525   9.81775  11.82600
P006            7.11375   6.67475  19.03050  13.57000

conditionId       RB            SB       
difficulty      easy   hard   easy   hard
participantId                            
P001            75.0  100.0  100.0   75.0
P003           100.0   75.0   75.0   75.0
P004            50.0   25.0   50.0  100.0
P005            75.0   75.0  100.0  100.0
P006            50.0   50.0   25.0   50.0

In [43]:
# Bearbeitungszeit: Swipe - Grid

diff_easy_duration = (
    duration_easy_hard[("SB", "easy")]
    - duration_easy_hard[("RB", "easy")]
)

diff_hard_duration = (
    duration_easy_hard[("SB", "hard")]
    - duration_easy_hard[("RB", "hard")]
)


# Accuracy: Swipe - Grid

diff_easy_accuracy = (
    accuracy_easy_hard[("SB", "easy")]
    - accuracy_easy_hard[("RB", "easy")]
)

diff_hard_accuracy = (
    accuracy_easy_hard[("SB", "hard")]
    - accuracy_easy_hard[("RB", "hard")]
)

In [44]:
interaction_duration = (
    diff_hard_duration - diff_easy_duration
)

interaction_accuracy = (
    diff_hard_accuracy - diff_easy_accuracy
)

### Interaction Contrast

Um zu untersuchen, ob sich der Unterschied zwischen Swipe und Grid zwischen den Easy- und Hard-Serien verändert, werden die Swipe-Grid-Differenz beider Blöcke verglichen.

Der Interaction Contrast wird berechnet als:

(Hard: Swipe - Grid) - (Easy: Swipe - Grid)

Ein Wert von 0 bedeutet, dass sich der Swipe-Grid-Unterschied zwischen den beiden Blöcken nicht verändert. Da Easy immer vor Hard präsentiert wurde, kann eine Veränderung nicht eindeutig allein dem Schwierigkeitsgrad zugeschrieben werden.


In [45]:
difficulty_descriptives = pd.DataFrame({
    "SB Easy Duration": duration_easy_hard[("SB", "easy")],
    "RB Easy Duration": duration_easy_hard[("RB", "easy")],
    "SB Hard Duration": duration_easy_hard[("SB", "hard")],
    "RB Hard Duration": duration_easy_hard[("RB", "hard")],

    "SB Easy Accuracy": accuracy_easy_hard[("SB", "easy")],
    "RB Easy Accuracy": accuracy_easy_hard[("RB", "easy")],
    "SB Hard Accuracy": accuracy_easy_hard[("SB", "hard")],
    "RB Hard Accuracy": accuracy_easy_hard[("RB", "hard")]
})

pd.DataFrame({
    "Mean": difficulty_descriptives.mean(),
    "Median": difficulty_descriptives.median(),
    "SD": difficulty_descriptives.std()
}).round(2)

,Mean,Median,SD
SB Easy Duration,13.18,11.86,5.33
RB Easy Duration,6.17,4.30,4.23
SB Hard Duration,7.90,6.48,3.63
RB Hard Duration,5.57,4.21,3.02
SB Easy Accuracy,61.90,75.00,26.95
RB Easy Accuracy,71.43,75.00,26.56
SB Hard Accuracy,75.00,75.00,25.00
RB Hard Accuracy,59.52,75.00,31.10


## Test des Einflusses der Schwierigkeit

Um zu prüfen, ob sich der Unterschied zwischen Swipe und Grid abhängig von der Schwierigkeit verändert, werden die Swipe-Grid-Differenzen der Easy- und Hard-Serien miteinander verglichen.

Zunächst wird mit dem Shapiro-Wilk-Test geprüft, ob die entsprechenden Interaction Contrasts normalverteilt sind.

In [46]:
print("Duration Interaction Contrast")
result_duration = stats.shapiro(interaction_duration)

print(f"W = {result_duration.statistic:.3f}")
print(f"p = {result_duration.pvalue:.3f}")
print()


print("Accuracy Interaction Contrast")
result_accuracy = stats.shapiro(interaction_accuracy)

print(f"W = {result_accuracy.statistic:.3f}")
print(f"p = {result_accuracy.pvalue:.3f}")

Duration Interaction Contrast
W = 0.935
p = 0.176

Accuracy Interaction Contrast
W = 0.932
p = 0.154


## Vergleich der Interaction Contrasts

Da die Shapiro-Wilk-Tests keinen signifikanten Hinweis auf eine Abweichung von der Normalverteilung ergeben, werden paired-Tests verwendet.

Dabei wird geprüft, ob sich die Differenz zwischen Swipe und Grid signifikant zwischen Easy- und Hard-Serien verändert.

In [47]:
# Einfluss der Schwierigkeit auf die Bearbeitungszeit

test_duration_interaction = stats.ttest_rel(
    diff_hard_duration,
    diff_easy_duration
)

# Einfluss der Schwierigkeit auf die Accuracy

test_accuracy_interaction = stats.ttest_rel(
    diff_hard_accuracy,
    diff_easy_accuracy
)

print("Duration")
print(f"t(20) = {test_duration_interaction.statistic:.3f}")
print(f"p = {test_duration_interaction.pvalue:.3f}")
print()

print("Accuracy")
print(f"t(20) = {test_accuracy_interaction.statistic:.3f}")
print(f"p = {test_accuracy_interaction.pvalue:.3f}")

Duration
t(20) = -3.422
p = 0.003

Accuracy
t(20) = 3.623
p = 0.002


In [48]:
d_duration_interaction = (
    interaction_duration.mean()
    / interaction_duration.std(ddof=1)
)

d_accuracy_interaction = (
    interaction_accuracy.mean()
    / interaction_accuracy.std(ddof=1)
)

print("Cohen's dz Duration:", round(d_duration_interaction, 3))
print("Cohen's dz Accuracy:", round(d_accuracy_interaction, 3))

Cohen's dz Duration: -0.747
Cohen's dz Accuracy: 0.791


## Korrektur für die zwei RQ2-Interaktionstests

Für RQ2 werden zwei Interaction Contrasts getestet. Einer für Bearbeitungszeit und einer für Accuracy. Daher werden die beiden p-Werte zusätzlich gemeinsam Bonferroni-korrigiert.


In [49]:
rq2_results = pd.DataFrame([
    {
        "Measure": "Duration",
        "t": test_duration_interaction.statistic,
        "df": len(interaction_duration) - 1,
        "p": test_duration_interaction.pvalue,
        "Cohen's dz": d_duration_interaction
    },
    {
        "Measure": "Accuracy",
        "t": test_accuracy_interaction.statistic,
        "df": len(interaction_accuracy) - 1,
        "p": test_accuracy_interaction.pvalue,
        "Cohen's dz": d_accuracy_interaction
    }
])

rq2_results["p_bonferroni"] = (
    rq2_results["p"] * 2
).clip(upper=1)

rq2_results["significant_bonferroni"] = (
    rq2_results["p_bonferroni"] < 0.05
)

rq2_results.round(3)


,Measure,t,df,p,Cohen's dz,p_bonferroni,significant_bonferroni
0,Duration,-3.422,20,0.003,-0.747,0.005,True
1,Accuracy,3.623,20,0.002,0.791,0.003,True


## Nichtparametrische Sensitivitätsanalyse der RQ2-Interaction-Contrasts

Da die Accuracy bei Easy und Hard jeweils nur auf vier Serien pro Person basiert und deshalb eine begrenzte Werteanzahl hat, werden die Interaction Contrasts zusätzlich mit Wilcoxon-Signed-Rank-Tests geprüft. Auch diese beiden Sensitivitätstests werden gemeinsam Bonferroni-korrigiert.


In [50]:
rq2_sensitivity = []

for label, hard_values, easy_values in [
    ("Duration", diff_hard_duration, diff_easy_duration),
    ("Accuracy", diff_hard_accuracy, diff_easy_accuracy)
]:
    test = stats.wilcoxon(
        hard_values,
        easy_values,
        alternative="two-sided",
        method="auto"
    )

    rq2_sensitivity.append({
        "Measure": label,
        "W": test.statistic,
        "p": test.pvalue
    })

rq2_sensitivity = pd.DataFrame(rq2_sensitivity)

rq2_sensitivity["p_bonferroni"] = (
    rq2_sensitivity["p"] * 2
).clip(upper=1)

rq2_sensitivity["significant_bonferroni"] = (
    rq2_sensitivity["p_bonferroni"] < 0.05
)

rq2_sensitivity.round(3)


,Measure,W,p,p_bonferroni,significant_bonferroni
0,Duration,36.0,0.004,0.009,True
1,Accuracy,15.0,0.005,0.010,True


## Explorative Sensitivitätsanalyse: erste Easy-Serie ausgeschlossen

Da die Easy-Serien immer vor den Hard-Serien bearbeitet wurden, kann besonders die erste Serie durch Eingewöhnung an die Interaktionstechnik beeinflusst sein. Deshalb wird explorativ geprüft, wie sich der Zeit-Interaction-Contrast verändert, wenn pro Person und Interface die jeweils erste bearbeitete Easy-Serie ausgeschlossen wird.

Diese Analyse war nicht vorab geplant und dient nur dazu, die Robustheit des Zeiteffekts gegenüber einem möglichen frühen Lern- oder Gewöhnungseffekt einzuschätzen.


In [51]:
# Erste tatsächlich bearbeitete Easy-Serie pro Person und Interface identifizieren
easy_rows = (
    series_level[series_level["difficulty"] == "easy"]
    .sort_values(["participantId", "conditionId", "start"])
)

first_easy_indices = (
    easy_rows
    .groupby(["participantId", "conditionId"])
    .head(1)
    .index
)

series_level_without_first_easy = series_level.drop(index=first_easy_indices)

# Durchschnittliche Serienzeit erneut pro Person / Interface / Schwierigkeit berechnen
duration_without_first_easy = (
    series_level_without_first_easy
    .groupby(
        ["participantId", "conditionId", "difficulty"],
        as_index=False
    )
    .agg(mean_durationSec=("durationSec", "mean"))
    .pivot(
        index="participantId",
        columns=["conditionId", "difficulty"],
        values="mean_durationSec"
    )
)

diff_easy_duration_reduced = (
    duration_without_first_easy[("SB", "easy")]
    - duration_without_first_easy[("RB", "easy")]
)

diff_hard_duration_reduced = (
    duration_without_first_easy[("SB", "hard")]
    - duration_without_first_easy[("RB", "hard")]
)

interaction_duration_reduced = (
    diff_hard_duration_reduced - diff_easy_duration_reduced
)

test_duration_reduced = stats.ttest_rel(
    diff_hard_duration_reduced,
    diff_easy_duration_reduced
)

print("Zeit-Interaction nach Ausschluss der ersten Easy-Serie")
print("Mean Interaction Contrast:", round(interaction_duration_reduced.mean(), 3))
print(f"t({len(interaction_duration_reduced)-1}) = {test_duration_reduced.statistic:.3f}")
print(f"p = {test_duration_reduced.pvalue:.4f}")
print("Bonferroni-Schwelle für zwei RQ2-Tests: alpha = 0.025")
print("Signifikant bei alpha=.025:", test_duration_reduced.pvalue < 0.025)


Zeit-Interaction nach Ausschluss der ersten Easy-Serie
Mean Interaction Contrast: -2.769
t(20) = -2.413
p = 0.0255
Bonferroni-Schwelle für zwei RQ2-Tests: alpha = 0.025
Signifikant bei alpha=.025: False


## Explorative Analyse der Interface-Reihenfolge bei Foto-Serien

Die Reihenfolge der vier Bedingungen wurde counterbalanced. Zusätzlich wird geprüft, ob der Zeitunterschied zwischen Swipe und Grid bei den Foto-Serien davon abhängt, welches der beiden Interfaces eine Person zuerst bearbeitet hat.

Die Analyse ist nicht hypothesenprüfend und beweist keinen Lerneffekt, sie dient dazu, einen möglichen Einfluss der Reihenfolge auf die Zeitmessung zu erkennen.


In [52]:
order_analysis = df[
    [
        "participantId",
        "SB_position",
        "RB_position",
        "SB_durationSec",
        "RB_durationSec"
    ]
].copy()

order_analysis["series_interface_order"] = np.where(
    order_analysis["SB_position"] < order_analysis["RB_position"],
    "Swipe first",
    "Grid first"
)

# Positiv = Swipe brauchte länger / Grid war schneller
order_analysis["duration_diff_swipe_minus_grid"] = (
    order_analysis["SB_durationSec"]
    - order_analysis["RB_durationSec"]
)

display(
    order_analysis
    .groupby("series_interface_order")["duration_diff_swipe_minus_grid"]
    .agg(["count", "mean", "median", "std"])
    .round(2)
)

swipe_first = order_analysis.loc[
    order_analysis["series_interface_order"] == "Swipe first",
    "duration_diff_swipe_minus_grid"
]

grid_first = order_analysis.loc[
    order_analysis["series_interface_order"] == "Grid first",
    "duration_diff_swipe_minus_grid"
]

order_test = stats.ttest_ind(
    swipe_first,
    grid_first,
    equal_var=False
)

print("Explorativer Welch-t-Test zwischen den Reihenfolgegruppen")
print(f"t = {order_test.statistic:.3f}")
print(f"p = {order_test.pvalue:.4f}")


,count,mean,median,std
series_interface_order,,,,
Grid first,11,12.17,17.57,25.29
Swipe first,10,64.45,58.81,23.84


Explorativer Welch-t-Test zwischen den Reihenfolgegruppen
t = 4.875
p = 0.0001
